In [ ]:
# Load in required libraries for analysis
library(tidyr)
library(ggplot2)
library(ggh4x)
library(dplyr)
library(RColorBrewer)
library(tidyselect)
library(tidyr)
library(stringr)
library(ggpubr)
library(labdsv)
library(vegan)
library(readr)
library(ape)
library(lme4)
library(lsmeans)
library(scales)
library(Polychrome)
library(reshape2)

In [ ]:
metadata <- read.csv("DOME-16S-Nasal-Metadata.csv")
metadata <- metadata %>%
  dplyr::mutate(Sample.ID = stringr::str_remove(Sample.ID, "^19-"))

In [ ]:
GENUS <- read.csv("DOME-16S-Taxa-Counts-Genus.csv"))

In [ ]:
asv_counts <- GENUS %>% dplyr::rename(Sample.ID = X)
relative_abundance_genus <- asv_counts %>%
  tibble::column_to_rownames("Sample.ID") %>%
  { 
    rs <- rowSums(.)
    rs[rs == 0] <- 1  # prevent divide by zero
    sweep(., 1, rs, "/")
  } %>%
  as.data.frame() %>%
  tibble::rownames_to_column("Sample.ID")


# Apply 0.05% cutoff
relative_abundance_genus <- relative_abundance_genus %>%
  dplyr::mutate(across(-Sample.ID, ~ ifelse(.x <= 0.0005, 0, .x)))

In [ ]:
top_20_long <- relative_abundance_genus %>%
  pivot_longer(-Sample.ID, names_to = "Genus", values_to = "Abundance") %>%
  mutate(
    Genus = ifelse(Genus %in% genus_order, Genus, "Other"),
    Genus = factor(Genus, levels = c(genus_order, "Other"))
  )

top_20_long <- top_20_long %>%
  mutate(Abundance = replace_na(Abundance * 100, 0))

top_20_long <- top_20_long %>%
  # Cohort based on Sample.ID prefix
  mutate(Cohort = case_when(
    startsWith(Sample.ID, "C") ~ "Cow",
    startsWith(Sample.ID, "D") ~ "Non-Farmer",
    startsWith(Sample.ID, "W") ~ "Farmer",
    TRUE ~ "Unknown"
  ))

top_20_long <- top_20_long %>%
  left_join(metadata %>% select(Sample.ID, Subject, Site, Season, Month, Year, Collection.Date, Sample.Type),
            by = "Sample.ID")

In [ ]:
# Compute unique samples across season x cohort
cohort_counts <- top_20_long %>%
  distinct(Sample.ID, Season, Cohort, Year) %>%
  group_by(Season, Cohort) %>%
  summarise(N = n(), .groups = "drop") %>%
  mutate(
    Cohort.Label = paste0(Cohort, " (N=", N, ")"),
    Season = factor(
      Season,
      levels = c("SPRING", "SUMMER", "FALL", "WINTER"),
      labels = c("Spring", "Summer", "Autumn", "Winter")
    )
  )

In [ ]:
# Extract abundance table
nasal_pcoa <- relative_abundance_genus %>%
  tibble::column_to_rownames("Sample.ID")

# Check for and remove empty rows
row_sums <- rowSums(nasal_pcoa, na.rm = TRUE)
empty_samples <- names(row_sums[row_sums == 0])

if (length(empty_samples) > 0) {
  message("Removing empty samples: ", paste(empty_samples, collapse = ", "))
}

nasal_pcoa_clean <- nasal_pcoa[row_sums > 0, ]

# Bray-Curtis distance
beta_div <- vegan::vegdist(nasal_pcoa_clean, index = "bray")

# PCoA
mds <- cmdscale(beta_div, k = 4, eig = TRUE)
mds_data <- as.data.frame(mds$points)
mds_data$Sample.ID <- rownames(mds_data)

# Merge with metadata
mds_data_merged <- left_join(mds_data, metadata, by = "Sample.ID")
mds_data_merged <- mds_data_merged %>%
  mutate(Cohort = case_when(
    startsWith(Sample.ID, "C") ~ "Cow",
    startsWith(Sample.ID, "D") ~ "Non-Farmer",
    startsWith(Sample.ID, "W") ~ "Farmer",
    TRUE ~ "Unknown"
  ))

eig_percentage <- round(100 * mds$eig / sum(mds$eig), 1)

In [ ]:
# Count occurrences of N
cohort_counts <- mds_data_merged %>%
  count(Cohort, name = "N")

label_map <- setNames(paste0(cohort_counts$Cohort, " (N=", cohort_counts$N, ")"),
                      cohort_counts$Cohort)

mds_data_merged <- mds_data_merged %>%
  mutate(Cohort_label = label_map[Cohort])

# Color palette
palette_colors <- c("#990006", "#B5B5B2", "#386EC2")

bray_curtis_pcoa_all <- ggplot(mds_data_merged, aes(x = V1, y = V2)) +
  geom_point(aes(color = Cohort_label), size = 1.5) +
  scale_color_manual(values = setNames(palette_colors, unique(mds_data_merged$Cohort_label))) +
  stat_ellipse(aes(group = Cohort_label, color = Cohort_label), linewidth = 1) +
  labs(
    x = paste0("PCoA 1 (", eig_percentage[1], "%)"),
    y = paste0("PCoA 2 (", eig_percentage[2], "%)"),
    color = "Cohort"
  ) +
  theme(
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
    panel.background = element_rect(fill = "white"),
    legend.title = element_blank(),
    legend.position = c(0.95, 0.05),         
    legend.justification = c("right", "bottom"), 
    legend.background = element_rect(fill = "white")
  ) +
  coord_equal()

In [ ]:
ggplot2::ggsave("DOME-16S-Bray-Curtis-PCoA-All.pdf", bray_curtis_pcoa_all, width = 9, height = 9, dpi = 320, units = "in")